# Phase 3: Machine Learning — Churn Prediction

Models: **Logistic Regression**, **Random Forest**, **XGBoost**

Metrics: Accuracy, Precision, Recall, F1, **ROC-AUC**

Run each cell in order. Requires Phase 1 cleaned CSV.

## 1. Imports & configuration

In [ ]:
%matplotlib inline

from pathlib import Path
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed — skipping XGBoost")

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("shap not installed — skipping SHAP plot")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "output").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CLEANED_CSV = PROJECT_ROOT / "output" / "phase1" / "cleaned_telco_churn.csv"
OUTPUT_DIR = PROJECT_ROOT / "output" / "phase3"
MODELS_DIR = OUTPUT_DIR / "models"
FIGURES_DIR = OUTPUT_DIR / "figures"
RANDOM_STATE = 42

for d in [OUTPUT_DIR, MODELS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

## 2. Load cleaned data

In [ ]:
df = pd.read_csv(CLEANED_CSV)
print(f"Loaded {len(df):,} rows")
df.head()

## 3. Prepare features & target

**Excluded (leakage / not for modeling):** Churn Score, Churn Reason, raw text columns, IDs

In [ ]:
EXCLUDE = {
    "CustomerID", "Churn Label", "Churn Value", "Churn_encoded",
    "Churn Score", "Churn Reason",
    "Country", "State", "City", "Zip Code", "Latitude", "Longitude",
    "Gender", "Senior Citizen", "Partner", "Dependents",
    "Phone Service", "Multiple Lines", "Internet Service",
    "Online Security", "Online Backup", "Device Protection",
    "Tech Support", "Streaming TV", "Streaming Movies",
    "Contract", "Paperless Billing", "Payment Method",
    "Tenure Group", "Payment Risk", "Risk Segment",
}

feature_cols = [c for c in df.columns if c not in EXCLUDE]
X = df[feature_cols]
y = df["Churn_encoded"]

print(f"{len(feature_cols)} features selected")
feature_cols

## 4. Train / test split (stratified)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Train churn rate: {y_train.mean():.2%}")
print(f"Test churn rate:  {y_test.mean():.2%}")

## 5. Define models

In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, class_weight="balanced")),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=12, random_state=RANDOM_STATE,
        class_weight="balanced", n_jobs=-1,
    ),
}
if HAS_XGB:
    models["XGBoost"] = xgb.XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=RANDOM_STATE, eval_metric="logloss",
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    )
list(models.keys())

## 6. Train & evaluate each model

In [ ]:
results = []
trained = {}

for name, model in models.items():
    print(f"\n{'='*50}\nTraining: {name}\n{'='*50}")
    model.fit(X_train, y_train)
    trained[name] = model

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    metrics = {
        "model": name,
        "accuracy": round(accuracy_score(y_test, y_pred), 4),
        "precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "f1": round(f1_score(y_test, y_pred, zero_division=0), 4),
        "roc_auc": round(roc_auc_score(y_test, y_prob), 4),
    }
    results.append({**metrics, "y_pred": y_pred, "y_prob": y_prob})
    print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn"]))
    joblib.dump(model, MODELS_DIR / f"{name.lower().replace(' ', '_')}.pkl")

metrics_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ("y_pred", "y_prob")} for r in results])
metrics_df.sort_values("roc_auc", ascending=False)

## 7. Confusion matrices

In [ ]:
for r in results:
    cm = confusion_matrix(y_test, r["y_pred"])
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["No Churn", "Churn"], yticklabels=["No Churn", "Churn"])
    ax.set_title(f"Confusion Matrix — {r['model']}")
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"confusion_matrix_{r['model'].lower().replace(' ', '_')}.png")
    plt.show()

## 8. ROC curves — model comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for r in results:
    fpr, tpr, _ = roc_curve(y_test, r["y_prob"])
    ax.plot(fpr, tpr, label=f"{r['model']} (AUC={r['roc_auc']:.3f})")
ax.plot([0, 1], [0, 1], "k--", label="Random")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — Model Comparison"); ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "roc_curves_comparison.png")
plt.show()

## 9. Feature importance

In [ ]:
for name, model in trained.items():
    if name == "Logistic Regression":
        imp = np.abs(model.named_steps["clf"].coef_[0])
    elif hasattr(model, "feature_importances_"):
        imp = model.feature_importances_
    else:
        continue
    imp_df = pd.DataFrame({"feature": feature_cols, "importance": imp})
    imp_df = imp_df.sort_values("importance", ascending=True).tail(15)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(imp_df["feature"], imp_df["importance"])
    ax.set_title(f"Top 15 Feature Importance — {name}")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"feature_importance_{name.lower().replace(' ', '_')}.png")
    plt.show()

## 10. SHAP explainability (XGBoost, if available)

In [ ]:
if HAS_SHAP and "XGBoost" in trained:
    explainer = shap.TreeExplainer(trained["XGBoost"])
    sample = X_test.sample(min(500, len(X_test)), random_state=RANDOM_STATE)
    shap_values = explainer.shap_values(sample)
    shap.summary_plot(shap_values, sample, feature_names=feature_cols, show=False)
    plt.title("SHAP Summary — XGBoost")
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "shap_summary_xgboost.png", bbox_inches="tight")
    plt.show()
else:
    print("SHAP or XGBoost not available — skip this cell.")

## 11. Churn probability scores & save outputs

In [ ]:
best_name = metrics_df.sort_values("roc_auc", ascending=False).iloc[0]["model"]
best_model = trained[best_name]
print(f"Best model by ROC-AUC: {best_name}")

predictions = df[["CustomerID", "Churn Label"]].copy()
predictions["churn_probability"] = best_model.predict_proba(X)[:, 1]
predictions["churn_prediction"] = (predictions["churn_probability"] >= 0.5).astype(int)
predictions["model_used"] = best_name

predictions.sort_values("churn_probability", ascending=False).head(10)

In [ ]:
metrics_df.to_csv(OUTPUT_DIR / "model_metrics.csv", index=False)
predictions.to_csv(OUTPUT_DIR / "churn_predictions.csv", index=False)
print(f"Saved metrics  -> {OUTPUT_DIR / 'model_metrics.csv'}")
print(f"Saved predictions -> {OUTPUT_DIR / 'churn_predictions.csv'}")
print(f"Saved models -> {MODELS_DIR}")